# EDA — `alerts_combined.json` · Exploración del dato crudo

Este es el Notebook del análisis exploratorio del archivo de entrada (crudo) antes de cualquier transformación del pipeline.
Cubre: estructura del JSON, tipos de dato, nulos, frecuencias, timestamps, relaciones entre columnas,
tablas pivote y distribución temporal.

**Este notebook es solo exploración — nunca es importado por el pipeline.**

In [1]:
import json
import re
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

warnings.filterwarnings("ignore")
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.float_format", "{:.2f}".format)

json_path = Path("..") / "data" / "raw" / "alerts_combined.json"
with open(json_path) as f:
    raw = json.load(f)

df = pd.DataFrame(raw)
print(f"Registros cargados : {len(df)}")
print(f"Columnas           : {list(df.columns)}")

Registros cargados : 458
Columnas           : ['ts', 'timestamp', 'source', 'priority', 'service', 'condition', 'threshold', 'policy', 'incidents', 'channel', 'error_type', 'error_message']


---
## 1 · Visión general del dataset

In [3]:
print(f"Shape: {df.shape}")
print()
print("dtypes del DataFrame (todos llegaron como object salvo incidents):")
print(df.dtypes)
print()
resumen = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "únicos": df.nunique(),
    "nulos": df.isnull().sum(),
    "cobertura_%": (1 - df.isnull().mean()).mul(100).round(1),
})
resumen

Shape: (458, 12)

dtypes del DataFrame (todos llegaron como object salvo incidents):
ts                object
timestamp         object
source            object
priority          object
service           object
condition         object
threshold         object
policy            object
incidents        float64
channel           object
error_type        object
error_message     object
dtype: object



,dtype,únicos,nulos,cobertura_%
ts,object,457,0,100.00
timestamp,object,455,0,100.00
source,object,2,0,100.00
priority,object,2,3,99.30
service,object,28,0,100.00
condition,object,23,0,100.00
threshold,object,27,3,99.30
policy,object,4,3,99.30
incidents,float64,6,3,99.30
channel,object,2,0,100.00


In [4]:
# Muestra de las 3 primeras y 3 últimas filas
pd.concat([df.head(3), df.tail(3)])

,ts,timestamp,source,priority,service,condition,threshold,policy,incidents,channel,error_type,error_message
0,1749647379.517959,2025-06-11T07:09:39-06:00,New Relic,high,Princess,High Application Response Time gral,>1800ms/5min,Golden Signals,1.00,sre,NaN,NaN
1,1749642905.830949,2025-06-11T05:55:05-06:00,New Relic,critical,Cerberus,High Application Error percentage,baseline/10min,Golden Signals,1.00,sre,NaN,NaN
2,1749642855.028159,2025-06-11T05:54:15-06:00,New Relic,critical,tesseract,Low Application Throughput,baseline/10min,Golden Signals,1.00,sre,NaN,NaN
455,1742000600,2026-03-27T16:06:33-06:00,New Relic,high,Hairs,Payments rejected hairs CX,>=5/5min,CX,1.00,monitoring-ops-cx,CARD_DECLINED,Tarjeta declinada. Intenta con otra tarjeta o método de pago.
456,1742000300,2026-03-27T16:12:02-06:00,New Relic,high,Hairs,Payments rejected hairs CX,>=5/5min,CX,1.00,monitoring-ops-cx,CARD_DECLINED,Tarjeta declinada. Intenta con otra tarjeta o método de pago.
457,1742000000,2026-03-27T16:21:20-06:00,New Relic,high,Hairs,Payments rejected hairs CX,>=5/5min,CX,1.00,monitoring-ops-cx,CARD_DECLINED,Tarjeta declinada. Intenta con otra tarjeta o método de pago.


---
## 2 · Los tres segmentos del archivo

El JSON concatena **3 exports distintos** con personalidades de reloj diferentes.
La regla operativa es por **canal** (no por índice) para ser robusta a datos nuevos:

| Segmento | Índices | Canal | Período real | Reloj confiable |
|---|---|---|---|---|
| 1 | 0–151 | sre | 7–11 jun 2025 | `ts` (epoch float) |
| 2 | 152–362 | sre | 26 may–4 jun 2025 | `ts` (epoch float) |
| 3 | 363–457 | monitoring-ops-cx | 24–27 mar 2026 | `timestamp` (ISO) |

In [5]:
seg1 = df.iloc[0:152].copy()
seg2 = df.iloc[152:363].copy()
seg3 = df.iloc[363:458].copy()

info = pd.DataFrame({
    "segmento"       : ["seg1 [0-151]", "seg2 [152-362]", "seg3 [363-457]"],
    "registros"      : [len(seg1), len(seg2), len(seg3)],
    "canal"          : [seg1["channel"].iloc[0], seg2["channel"].iloc[0], seg3["channel"].iloc[0]],
    "periodo_real"   : ["7-11 jun 2025", "26 may-4 jun 2025", "24-27 mar 2026"],
    "reloj_confiable": ["ts (epoch float)", "ts (epoch float)", "timestamp (ISO)"],
})
print(info.to_string(index=False))

print()
print("Columnas extra en seg3 vs seg1/seg2:")
extra = set(seg3.columns[seg3.notna().any()]) - set(seg1.columns[seg1.notna().any()])
print(f"  {sorted(extra)}  ← solo existen con datos en el canal cx")

print()
print("Columnas con nulos SOLO en seg3:")
for col in ["priority", "threshold", "policy", "incidents"]:
    n1 = seg1[col].isna().sum()
    n2 = seg2[col].isna().sum()
    n3 = seg3[col].isna().sum()
    if n3 > 0:
        print(f"  {col}: seg1={n1}, seg2={n2}, seg3={n3}")

      segmento  registros             canal      periodo_real  reloj_confiable
  seg1 [0-151]        152               sre     7-11 jun 2025 ts (epoch float)
seg2 [152-362]        211               sre 26 may-4 jun 2025 ts (epoch float)
seg3 [363-457]         95 monitoring-ops-cx    24-27 mar 2026  timestamp (ISO)

Columnas extra en seg3 vs seg1/seg2:
  ['error_message', 'error_type']  ← solo existen con datos en el canal cx

Columnas con nulos SOLO en seg3:


---
## 3 · Frecuencias de columnas categóricas

In [6]:
def freq_table(series, name=None):
    """Tabla de frecuencias absoluta + porcentaje para una Serie."""
    vc = series.value_counts(dropna=False)
    t = vc.rename("n").to_frame()
    t["pct"] = (t["n"] / len(series) * 100).round(1)
    t.index.name = name or series.name
    return t

print("=== channel ===")
print(freq_table(df["channel"]))
print()
print("=== source ===")
print(freq_table(df["source"]))

=== channel ===
                     n   pct
channel                     
sre                363 79.30
monitoring-ops-cx   95 20.70

=== source ===
                 n   pct
source                  
New Relic      455 99.30
PayPal Status    3  0.70


In [8]:
print("=== priority (incluyendo nulos) ===")
print(freq_table(df["priority"]))
print()
print("=== policy (incluyendo nulos) ===")
print(freq_table(df["policy"]))

=== priority (incluyendo nulos) ===
            n   pct
priority           
high      234 51.10
critical  221 48.30
None        3  0.70

=== policy (incluyendo nulos) ===
                    n   pct
policy                     
Golden Signals    254 55.50
Parco2.0 strict    95 20.70
CX                 95 20.70
Up_Satatus_Parco   11  2.40
None                3  0.70


In [9]:
print("=== condition — las 23 únicas (ordenadas por frecuencia) ===")
print(freq_table(df["condition"]).to_string())

=== condition — las 23 únicas (ordenadas por frecuencia) ===
                                       n   pct
condition                                     
Payments rejected hairs CX            95 20.70
high request count with status 500    91 19.90
Throughput high general               58 12.70
High Application Error percentage     44  9.60
Status code 500 counted request       31  6.80
Parco 2.0 Nodes CPU Usage             23  5.00
RDS CPU Usage gral                    19  4.10
Carts Throughput high                 17  3.70
global traffic alert                  12  2.60
Cerberus Throughput High              11  2.40
Parco APIs status - locations failed  11  2.40
High Application Response Time gral   11  2.40
Apdex score                           11  2.40
Low Application Throughput             9  2.00
Payments rejected hairs                4  0.90
SMS Alert                              4  0.90
External Scan Alert                    1  0.20
Intermittent Disruption - RESOLVED     1  0.20

In [10]:
print("=== service — los 28 únicos (ordenados por frecuencia) ===")
print(freq_table(df["service"]).to_string())

=== service — los 28 únicos (ordenados por frecuencia) ===
                        n   pct
service                        
Hairs                 124 27.10
Orchestrator          104 22.70
Wallet_2.0             31  6.80
tesseract              29  6.30
Carts                  28  6.10
Users                  26  5.70
Cerberus               20  4.40
Transaction query      12  2.60
i-058689de5ec046291    11  2.40
Princess               11  2.40
Wiki                    9  2.00
i-0d26dd24e2a69bff0     7  1.50
data-team               7  1.50
Access                  6  1.30
new-parco-instance-1    6  1.30
Kraken                  5  1.10
demo2                   4  0.90
Gigante                 3  0.70
i-04acbab68c9357917     2  0.40
Peajero                 2  0.40
i-0d568d70a0847d5b6     2  0.40
new-parco-instance-5    2  0.40
Chargehound             2  0.40
PayPal                  1  0.20
Web Page Parco          1  0.20
Wordpress web page      1  0.20
Invoice                 1  0.20
Princes      

---
## 4 · Canal cx — `error_type`, `error_message` y procesador derivado

Estas columnas **solo existen en monitoring-ops-cx** (seg3). No son missingness en sre — son esquemas de fuente distintos.

In [26]:
cx = df[df["channel"] == "monitoring-ops-cx"].copy()
print(f"Registros cx: {len(cx)}")
print(f"Cobertura error_type en cx: {cx['error_type'].notna().mean()*100:.1f}%")
print()
print("=== error_type ===")
print(freq_table(cx["error_type"]))
print()
print("=== error_message (valores únicos) ===")
print(freq_table(cx["error_message"]).to_string())

Registros cx: 95
Cobertura error_type en cx: 100.0%

=== error_type ===
                       n   pct
error_type                    
INSUFFICIENT_FUNDS    28 29.50
IMPOSSIBLE_TO_CHARGE  25 26.30
CARD_DECLINED         23 24.20
BANK_REJECTED         12 12.60
INSTRUMENT_DECLINED    4  4.20
BLOCKED_CREDIT_CARD    1  1.10
PAYPAL_UNAVAILABLE     1  1.10
MERCADOPAGO_ERROR      1  1.10

=== error_message (valores únicos) ===
                                                                 n   pct
error_message                                                           
INSUFFICIENT_FUNDS                                              27 28.40
IMPOSSIBLE_TO_CHARGE                                            25 26.30
Tarjeta declinada. Intenta con otra tarjeta o método de pago.   22 23.20
El banco emisor rechazó el pago sin más detalles (Conekta)      12 12.60
INSTRUMENT_DECLINED                                              4  4.20
Fondos insuficientes                                             1 

In [27]:
def extraer_procesador(msg):
    """Deriva el procesador de pago desde error_message."""
    if pd.isna(msg):
        return None
    m = msg.lower()
    if "conekta" in m:
        return "Conekta"
    if "mercadopago" in m or "mercado" in m:
        return "Mercadopago"
    if "paypal" in m:
        return "PayPal"
    return "Otro"

cx["procesador"] = cx["error_message"].apply(extraer_procesador)

print("=== procesador (derivado de error_message) ===")
print(freq_table(cx["procesador"]))
print()
print("=== Pivot: error_type × procesador ===")
print(pd.crosstab(cx["error_type"], cx["procesador"], margins=True))

=== procesador (derivado de error_message) ===
              n   pct
procesador           
Otro         81 85.30
Conekta      12 12.60
PayPal        1  1.10
Mercadopago   1  1.10

=== Pivot: error_type × procesador ===
procesador            Conekta  Mercadopago  Otro  PayPal  All
error_type                                                   
BANK_REJECTED              12            0     0       0   12
BLOCKED_CREDIT_CARD         0            0     1       0    1
CARD_DECLINED               0            0    23       0   23
IMPOSSIBLE_TO_CHARGE        0            0    25       0   25
INSTRUMENT_DECLINED         0            0     4       0    4
INSUFFICIENT_FUNDS          0            0    28       0   28
MERCADOPAGO_ERROR           0            1     0       0    1
PAYPAL_UNAVAILABLE          0            0     0       1    1
All                        12            1    81       1   95


---
## 5 · Columna numérica: `incidents`

En New Relic, `incidents` es cuántos disparos reales empaquetó una notificación. Los 3 registros con null son PayPal Status (fuente distinta).

In [28]:
print("=== estadísticas de incidents ===")
print(df["incidents"].describe())
print()
print("=== distribución de valores (incluyendo nulos) ===")
print(freq_table(df["incidents"]))
print()
suma_con_impute = df["incidents"].fillna(1).astype(int).sum()
print(f"Suma total con null→1 (imputación mínima): {suma_con_impute}")
print(f"  → 664 si se excluyen los 3 PayPal (no New Relic)")
print(f"  → 667 si se imputan como 1")

=== estadísticas de incidents ===
count   455.00
mean      1.46
std       0.84
min       1.00
25%       1.00
50%       1.00
75%       2.00
max       7.00
Name: incidents, dtype: float64

=== distribución de valores (incluyendo nulos) ===
             n   pct
incidents           
1.00       298 65.10
2.00       131 28.60
3.00        14  3.10
4.00         6  1.30
6.00         4  0.90
NaN          3  0.70
7.00         2  0.40

Suma total con null→1 (imputación mínima): 667
  → 664 si se excluyen los 3 PayPal (no New Relic)
  → 667 si se imputan como 1


In [29]:
# ¿Los registros con >1 incident son rachas o picos aislados?
altas = df[df["incidents"] >= 4][["channel", "service", "condition", "incidents", "priority"]]
print(f"Registros con incidents >= 4: {len(altas)}")
print(altas.sort_values("incidents", ascending=False).to_string(index=False))

Registros con incidents >= 4: 12
channel             service                 condition  incidents priority
    sre i-058689de5ec046291 Parco 2.0 Nodes CPU Usage       7.00 critical
    sre               Hairs   Throughput high general       7.00 critical
    sre               Carts     Carts Throughput high       6.00 critical
    sre           tesseract   Throughput high general       6.00 critical
    sre           tesseract   Throughput high general       6.00 critical
    sre               Hairs   Throughput high general       6.00 critical
    sre            Cerberus  Cerberus Throughput High       4.00 critical
    sre               Carts     Carts Throughput high       4.00 critical
    sre            Cerberus  Cerberus Throughput High       4.00 critical
    sre i-058689de5ec046291 Parco 2.0 Nodes CPU Usage       4.00 critical
    sre            Cerberus  Cerberus Throughput High       4.00 critical
    sre   Transaction query      global traffic alert       4.00 critical


---
## 6 · Timestamps — dos relojes, dos canales

Hay **dos campos de tiempo** (`ts` epoch float y `timestamp` ISO) pero no son intercambiables:
- `ts` en sre: epoch Unix en float (confiable)
- `timestamp` en sre seg2: tiene desfases erráticos de hasta 17 h (no confiable)
- `ts` en cx: sintético, incremento fijo de exactamente 300 s (no es el tiempo real)
- `timestamp` en cx: ISO con timezone (confiable)

In [30]:
sre_df = df[df["channel"] == "sre"].copy()
cx_df  = df[df["channel"] == "monitoring-ops-cx"].copy()

# SRE: ts epoch → datetime
sre_df["dt"] = (
    pd.to_datetime(sre_df["ts"].astype(float), unit="s", utc=True)
    .dt.tz_convert("America/Mexico_City")
)
# CX: timestamp ISO → datetime
cx_df["dt"] = (
    pd.to_datetime(cx_df["timestamp"], utc=True)
    .dt.tz_convert("America/Mexico_City")
)

print("=== SRE — rango vía ts (epoch, confiable) ===")
print(f"  min : {sre_df['dt'].min()}")
print(f"  max : {sre_df['dt'].max()}")
print(f"  días: {(sre_df['dt'].max() - sre_df['dt'].min()).days}")
print()
print("=== CX — rango vía timestamp ISO (confiable) ===")
print(f"  min : {cx_df['dt'].min()}")
print(f"  max : {cx_df['dt'].max()}")
print(f"  días: {(cx_df['dt'].max() - cx_df['dt'].min()).days}")

=== SRE — rango vía ts (epoch, confiable) ===
  min : 2025-05-26 15:06:22-06:00
  max : 2025-06-11 07:09:39.517959118-06:00
  días: 15

=== CX — rango vía timestamp ISO (confiable) ===
  min : 2026-03-24 12:54:07-06:00
  max : 2026-03-27 16:21:20-06:00
  días: 3


In [31]:
# CX ts: verificar que es sintético (gap fijo)
cx_ts = cx_df["ts"].astype(float)
diffs = cx_ts.diff().abs().dropna()
print("=== CX ts: distribución de diferencias entre registros consecutivos ===")
print(diffs.value_counts().rename("n").to_frame())
print("→ ts en cx es sintético: gap fijo de exactamente 300 s (5 min) en los 94 intervalos")

=== CX ts: distribución de diferencias entre registros consecutivos ===
         n
ts        
300.00  94
→ ts en cx es sintético: gap fijo de exactamente 300 s (5 min) en los 94 intervalos


In [15]:
# SRE seg2 [152-362]: desfase entre epoch y timestamp ISO
sre2 = df.iloc[152:363].copy()
sre2["dt_epoch"] = pd.to_datetime(sre2["ts"].astype(float), unit="s", utc=True)
sre2["dt_iso"]   = pd.to_datetime(sre2["timestamp"], utc=True)
sre2["drift_h"]  = (sre2["dt_epoch"] - sre2["dt_iso"]).dt.total_seconds() / 3600

print("=== SRE seg2 [152-362]: desfase epoch vs ISO (horas) ===")
print(sre2["drift_h"].describe())
print()
print("Distribución del desfase (en horas enteras):")
print(sre2["drift_h"].round(0).value_counts().sort_index())
print()
print("→ El campo ISO tiene desfases erráticos de -6 h a +17 h → se descarta para sre")
print("→ Regla final: sre usa ts (epoch), cx usa timestamp (ISO)")

=== SRE seg2 [152-362]: desfase epoch vs ISO (horas) ===
count   211.00
mean     -1.90
std       3.21
min      -6.00
25%      -3.67
50%      -3.33
75%      -1.00
max      17.00
Name: drift_h, dtype: float64

Distribución del desfase (en horas enteras):
drift_h
-6.00     7
-5.00     7
-4.00    55
-3.00    76
-2.00     4
-1.00    12
0.00      6
2.00     20
3.00     15
4.00      6
6.00      1
16.00     1
17.00     1
Name: count, dtype: int64

→ El campo ISO tiene desfases erráticos de -6 h a +17 h → se descarta para sre
→ Regla final: sre usa ts (epoch), cx usa timestamp (ISO)


In [32]:
# ¿Cuántos ts únicos hay? (detectar colisiones)
print(f"ts únicos en total  : {df['ts'].nunique()} de {len(df)}")
print(f"timestamp únicos    : {df['timestamp'].nunique()} de {len(df)}")
ts_dup = df[df["ts"].duplicated(keep=False)]
print(f"\nRegistros con ts duplicado: {len(ts_dup)}")
if len(ts_dup) > 0:
    print(ts_dup[["ts", "channel", "service", "condition"]].to_string(index=False))

ts únicos en total  : 457 de 458
timestamp únicos    : 455 de 458

Registros con ts duplicado: 2
        ts channel   service               condition
1748724165     sre     Hairs Throughput high general
1748724165     sre tesseract Throughput high general


---
## 7 · Columna `threshold` — parsing de reglas

Cada threshold encapsula 3 dimensiones: **tipo** (estática o anomalía/baseline), **dirección** (sobre o bajo el umbral) y **ventana de evaluación** en minutos.

In [17]:
print("=== Valores únicos de threshold (incluyendo nulos) ===")
print(freq_table(df["threshold"]).to_string())

=== Valores únicos de threshold (incluyendo nulos) ===
                 n   pct
threshold               
>=5/5min        95 20.70
>55/5min        58 12.70
>2700/5min      53 11.60
baseline/10min  53 11.60
>60/5min        33  7.20
>20/5min        30  6.60
>50/5min        23  5.00
>2800/5min      16  3.50
>55%/10min      14  3.10
1+ locations    11  2.40
>1800ms/5min    11  2.40
>6000/5min      11  2.40
>3500/5min      10  2.20
<0.5/3min        7  1.50
>3200/5min       5  1.10
>60%/10min       5  1.10
<0.6/3min        4  0.90
>40/5min         4  0.90
>21/5min         4  0.90
NaN              3  0.70
>30/5min         1  0.20
>2800ms/5min     1  0.20
>4/5min          1  0.20
>3800/5min       1  0.20
>6500/5min       1  0.20
>3400/5min       1  0.20
>130/5min        1  0.20
>10%/5min        1  0.20


In [38]:
def parse_threshold(t):
    if pd.isna(t):
        return {"tipo_regla": None, "direccion": None, "ventana_eval_min": None, "raw": t}
    tipo = "anomalia" if "baseline" in t else "estatica"
    if ">" in t:
        dir_ = "sobre"
    elif "<" in t:
        dir_ = "bajo"
    else:
        dir_ = "neutral"
    m = re.search(r"/(\d+)min", t)
    ventana = int(m.group(1)) if m else None
    return {"tipo_regla": tipo, "direccion": dir_, "ventana_eval_min": ventana, "raw": t}

th_uniq = df["threshold"].dropna().drop_duplicates()
th_df = pd.DataFrame([parse_threshold(t) for t in th_uniq])

print("=== Reglas únicas parseadas ===")
print(th_df.sort_values(["tipo_regla", "direccion", "ventana_eval_min"]).to_string(index=False))

=== Reglas únicas parseadas ===
tipo_regla direccion  ventana_eval_min            raw
  anomalia   neutral             10.00 baseline/10min
  estatica      bajo              3.00      <0.5/3min
  estatica      bajo              3.00      <0.6/3min
  estatica   neutral               NaN   1+ locations
  estatica     sobre              5.00   >1800ms/5min
  estatica     sobre              5.00     >3500/5min
  estatica     sobre              5.00       >60/5min
  estatica     sobre              5.00       >55/5min
  estatica     sobre              5.00   >2800ms/5min
  estatica     sobre              5.00     >6000/5min
  estatica     sobre              5.00       >50/5min
  estatica     sobre              5.00        >4/5min
  estatica     sobre              5.00      >10%/5min
  estatica     sobre              5.00     >2800/5min
  estatica     sobre              5.00       >20/5min
  estatica     sobre              5.00       >30/5min
  estatica     sobre              5.00     >2700/5

In [39]:
print("=== Resumen: tipo_regla × dirección (sobre reglas únicas) ===")
print(pd.crosstab(th_df["tipo_regla"], th_df["direccion"], margins=True))
print()
print("=== Ventanas de evaluación (minutos) ===")
print(th_df["ventana_eval_min"].value_counts().sort_index().rename("n_reglas_únicas").to_frame())
print()
# Aplicado al dataset completo (pesado por frecuencia)
df_th = df["threshold"].dropna().apply(lambda t: parse_threshold(t)["tipo_regla"])
print("=== Distribución tipo_regla sobre las 455 alertas con threshold ===")
print(df_th.value_counts().rename("n_alertas").to_frame())

=== Resumen: tipo_regla × dirección (sobre reglas únicas) ===
direccion   bajo  neutral  sobre  All
tipo_regla                           
anomalia       0        1      0    1
estatica       2        1     23   26
All            2        2     23   27

=== Ventanas de evaluación (minutos) ===
                  n_reglas_únicas
ventana_eval_min                 
3.00                            2
5.00                           21
10.00                           3

=== Distribución tipo_regla sobre las 455 alertas con threshold ===
           n_alertas
threshold           
estatica         402
anomalia          53


---
## 8 · Nulos — contexto por fuente

Los nulos no son todos iguales: algunos son **ausencia estructural** (columna no existe en esa fuente) y otros son **missingness real**.

In [45]:
nulos = df.isnull().sum()
nulos_df = nulos[nulos > 0].rename("n_nulos").to_frame()
nulos_df["pct"] = (nulos_df["n_nulos"] / len(df) * 100).round(1)
nulos_df["tipo"] = ""
nulos_df["origen_o_causa"] = ""

# error_type / error_message: ausencia estructural
for c in ["error_type", "error_message"]:
    nulos_df.loc[c, "tipo"] = "ausencia estructural"
    nulos_df.loc[c, "origen_o_causa"] = "Solo existe en cx; ausente en sre es esperado (distinto esquema de fuente)"

# Los otros 4: los 3 registros PayPal Status
for c in ["priority", "threshold", "policy", "incidents"]:
    nulos_df.loc[c, "tipo"] = "fuente sin campo"
    nulos_df.loc[c, "origen_o_causa"] = "Los 3 registros PayPal Status (source≠New Relic) no tienen este campo estructurado"

print(nulos_df.to_string())
print()
print("→ Completitud real para datos New Relic: 100% en todas las columnas propias")

               n_nulos   pct                  tipo                                                                      origen_o_causa
priority             3  0.70      fuente sin campo  Los 3 registros PayPal Status (source≠New Relic) no tienen este campo estructurado
threshold            3  0.70      fuente sin campo  Los 3 registros PayPal Status (source≠New Relic) no tienen este campo estructurado
policy               3  0.70      fuente sin campo  Los 3 registros PayPal Status (source≠New Relic) no tienen este campo estructurado
incidents            3  0.70      fuente sin campo  Los 3 registros PayPal Status (source≠New Relic) no tienen este campo estructurado
error_type         363 79.30  ausencia estructural          Solo existe en cx; ausente en sre es esperado (distinto esquema de fuente)
error_message      363 79.30  ausencia estructural          Solo existe en cx; ausente en sre es esperado (distinto esquema de fuente)

→ Completitud real para datos New Relic: 100% en todas

In [46]:
# Verificar: los 3 nulos de priority son exactamente los PayPal Status
paypal = df[df["priority"].isna()][["source", "condition", "channel", "priority", "threshold", "policy", "incidents"]]
print("Registros con priority=null:")
print(paypal.to_string(index=True))

Registros con priority=null:
            source                           condition channel priority threshold policy  incidents
7    PayPal Status  Intermittent Disruption - RESOLVED     sre     None       NaN   None        NaN
16   PayPal Status   Intermittent Disruption - INITIAL     sre     None       NaN   None        NaN
152  PayPal Status               Scheduled Maintenance     sre     None       NaN   None        NaN


---
## 9 · Duplicados exactos

In [47]:
n_dup = df.duplicated().sum()
print(f"Duplicados byte a byte (todas las columnas): {n_dup}")

# Cuasi-duplicados por clave de negocio
clave = ["channel", "service", "condition", "ts"]
n_cuasi = df.duplicated(subset=clave).sum()
print(f"Cuasi-duplicados (channel+service+condition+ts): {n_cuasi}")

# ¿Hay ts repetidos?
ts_dup = df[df["ts"].duplicated(keep=False)]
print(f"\nts repetidos en el dataset: {len(ts_dup)}")
if len(ts_dup) > 0:
    print(ts_dup[["ts", "channel", "service", "condition", "priority"]].to_string(index=False))

print()
print("→ 0 duplicados exactos. El pipeline no necesita deduplicación por igualdad byte a byte.")
print("→ La deduplicación real es por fingerprint+ventana temporal (agrupa ráfagas de la misma alerta).")

Duplicados byte a byte (todas las columnas): 0
Cuasi-duplicados (channel+service+condition+ts): 0

ts repetidos en el dataset: 2
        ts channel   service               condition priority
1748724165     sre     Hairs Throughput high general     high
1748724165     sre tesseract Throughput high general     high

→ 0 duplicados exactos. El pipeline no necesita deduplicación por igualdad byte a byte.
→ La deduplicación real es por fingerprint+ventana temporal (agrupa ráfagas de la misma alerta).


---
## 10 · Relaciones entre columnas

In [51]:
# condition → policy: ¿es 1-to-1?
print("=== condition → policy: ¿1-to-1? ===")
rel = df.dropna(subset=["policy"]).groupby("condition")["policy"].nunique()
multi = rel[rel > 1]
if len(multi) == 0:
    print("✓ Partición perfecta: cada condition tiene exactamente 1 policy")
else:
    print(f"✗ {len(multi)} conditions tienen múltiples policies:")
    print(multi)

print()
print("Mapa completo condition → policy:")
mapa = (
    df.dropna(subset=["policy"])
    .groupby("condition")["policy"]
    .first()
    .reset_index()
    .rename(columns={"policy": "policy_única"})
)
print(mapa.to_string(index=False))

=== condition → policy: ¿1-to-1? ===
✓ Partición perfecta: cada condition tiene exactamente 1 policy

Mapa completo condition → policy:
                           condition     policy_única
                         Apdex score   Golden Signals
               Carts Throughput high   Golden Signals
            Cerberus Throughput High   Golden Signals
               Error percentage high   Golden Signals
                 External Scan Alert   Golden Signals
   High Application Error percentage   Golden Signals
 High Application Response Time gral   Golden Signals
                  High response time   Golden Signals
          Low Application Throughput   Golden Signals
           Parco 2.0 Nodes CPU Usage   Golden Signals
Parco APIs status - locations failed Up_Satatus_Parco
             Payments rejected hairs  Parco2.0 strict
          Payments rejected hairs CX               CX
                  RDS CPU Usage gral   Golden Signals
                           SMS Alert   Golden Signals


In [52]:
# policy → channel: ¿partición perfecta?
print("=== policy → channel: ¿partición perfecta? ===")
rel2 = df.dropna(subset=["policy"]).groupby("policy")["channel"].nunique()
multi2 = rel2[rel2 > 1]
if len(multi2) == 0:
    print("✓ Partición perfecta: cada policy pertenece a exactamente 1 channel")
else:
    print(f"✗ {len(multi2)} policies aparecen en múltiples channels:")
    print(multi2)

print()
print("Pivot policy × channel (con totales):")
print(pd.crosstab(df["policy"], df["channel"], margins=True))

=== policy → channel: ¿partición perfecta? ===
✓ Partición perfecta: cada policy pertenece a exactamente 1 channel

Pivot policy × channel (con totales):
channel           monitoring-ops-cx  sre  All
policy                                       
CX                               95    0   95
Golden Signals                    0  254  254
Parco2.0 strict                   0   95   95
Up_Satatus_Parco                  0   11   11
All                              95  360  455


In [54]:
# Pivot: service × priority (canal sre)
sre_df2 = df[df["channel"] == "sre"].copy()
pivot_sp = pd.crosstab(sre_df2["service"], sre_df2["priority"], margins=True)
pivot_sp = pivot_sp.sort_values("All", ascending=False)
pivot_sp["pct_critical"] = (pivot_sp.get("critical", 0) / pivot_sp["All"] * 100).round(1)
print("=== Pivot: service × priority (canal sre, n=363) ===")
print(pivot_sp.to_string())

=== Pivot: service × priority (canal sre, n=363) ===
priority              critical  high  All  pct_critical
service                                                
All                        221   139  360         61.40
Orchestrator                89    15  104         85.60
Wallet_2.0                   2    29   31          6.50
Hairs                       12    17   29         41.40
tesseract                   12    17   29         41.40
Carts                       20     8   28         71.40
Users                        6    20   26         23.10
Cerberus                    19     1   20         95.00
Transaction query           11     1   12         91.70
i-058689de5ec046291          4     7   11         36.40
Princess                     3     8   11         27.30
Wiki                         9     0    9        100.00
i-0d26dd24e2a69bff0          2     5    7         28.60
data-team                    7     0    7        100.00
new-parco-instance-1         3     3    6         5

In [26]:
# Pivot: condition × channel
print("=== Pivot: condition × channel (con totales) ===")
cxch = pd.crosstab(df["condition"], df["channel"], margins=True)
print(cxch.sort_values("All", ascending=False).to_string())
print()
conditions_solo_sre = cxch[(cxch["monitoring-ops-cx"] == 0) & (cxch["sre"] > 0)].index.drop("All", errors="ignore")
conditions_solo_cx  = cxch[(cxch["sre"] == 0) & (cxch["monitoring-ops-cx"] > 0)].index.drop("All", errors="ignore")
print(f"Conditions exclusivos de sre: {len(conditions_solo_sre)}")
print(f"Conditions exclusivos de cx : {len(conditions_solo_cx)}")
print(f"Conditions compartidos       : 0")

=== Pivot: condition × channel (con totales) ===
channel                               monitoring-ops-cx  sre  All
condition                                                        
All                                                  95  363  458
Payments rejected hairs CX                           95    0   95
high request count with status 500                    0   91   91
Throughput high general                               0   58   58
High Application Error percentage                     0   44   44
Status code 500 counted request                       0   31   31
Parco 2.0 Nodes CPU Usage                             0   23   23
RDS CPU Usage gral                                    0   19   19
Carts Throughput high                                 0   17   17
global traffic alert                                  0   12   12
Apdex score                                           0   11   11
High Application Response Time gral                   0   11   11
Cerberus Throughput High   

In [27]:
# Pivot: condition × priority (canal sre)
print("=== Pivot: condition × priority (canal sre) ===")
cp = pd.crosstab(sre_df2["condition"], sre_df2["priority"], margins=True)
cp = cp.sort_values("All", ascending=False)
cp["pct_critical"] = (cp.get("critical", 0) / cp["All"] * 100).round(1)
print(cp.to_string())

=== Pivot: condition × priority (canal sre) ===


priority                              critical  high  All  pct_critical
condition                                                              
All                                        221   139  360         61.40
high request count with status 500          76    15   91         83.50
Throughput high general                     13    45   58         22.40
High Application Error percentage           44     0   44        100.00
Status code 500 counted request              2    29   31          6.50
Parco 2.0 Nodes CPU Usage                    9    14   23         39.10
RDS CPU Usage gral                          14     5   19         73.70
Carts Throughput high                        9     8   17         52.90
global traffic alert                        11     1   12         91.70
Apdex score                                 11     0   11        100.00
High Application Response Time gral          1    10   11          9.10
Cerberus Throughput High                    10     1   11       

---
## 11 · Fingerprint `{service}::{condition}` — análisis de recurrencia cruda

In [28]:
df["fingerprint"] = df["service"] + "::" + df["condition"]
print(f"Fingerprints únicos: {df['fingerprint'].nunique()}")
print()
fp_freq = df["fingerprint"].value_counts()
print("=== Top 20 fingerprints más frecuentes ===")
print(fp_freq.head(20).rename("n").to_frame().assign(pct=lambda x: (x.n / len(df) * 100).round(1)).to_string())

Fingerprints únicos: 47

=== Top 20 fingerprints más frecuentes ===
                                                   n   pct
fingerprint                                               
Hairs::Payments rejected hairs CX                 95 20.70
Orchestrator::high request count with status 500  91 19.90
Wallet_2.0::Status code 500 counted request       31  6.80
tesseract::Throughput high general                20  4.40
Hairs::Throughput high general                    19  4.10
Users::Throughput high general                    19  4.10
Carts::Carts Throughput high                      17  3.70
Orchestrator::High Application Error percentage   13  2.80
Transaction query::global traffic alert           12  2.60
Carts::Apdex score                                11  2.40
Cerberus::Cerberus Throughput High                11  2.40
i-058689de5ec046291::Parco 2.0 Nodes CPU Usage    11  2.40
Wiki::Parco APIs status - locations failed         9  2.00
Princess::High Application Response Time gral  

In [29]:
singletons = fp_freq[fp_freq == 1]
print(f"Fingerprints con 1 sola alerta (singletones): {len(singletons)}")
print(singletons.rename("n").to_frame().to_string())
print()
# Distribución de frecuencias de fingerprints
print("=== Distribución: ¿cuántos fingerprints tienen N alertas? ===")
dist = fp_freq.value_counts().sort_index().rename("n_fingerprints")
dist.index.name = "n_alertas"
print(dist.to_frame().to_string())

Fingerprints con 1 sola alerta (singletones): 15
                                                      n
fingerprint                                            
Kraken::High Application Response Time gral           1
Gigante::Low Application Throughput                   1
Users::login alert                                    1
Kraken::High response time                            1
Invoice::High Application Error percentage            1
Wordpress web page::Parco 2.0 Nodes CPU Usage         1
Web Page Parco::Parco APIs status - locations failed  1
Princes::Parco APIs status - locations failed         1
PayPal::Scheduled Maintenance                         1
tesseract::Low Application Throughput                 1
Cerberus::Low Application Throughput                  1
Chargehound::Intermittent Disruption - INITIAL        1
Kraken::Error percentage high                         1
tesseract::External Scan Alert                        1
Chargehound::Intermittent Disruption - RESOLVED       1

---
## 12 · Distribución temporal

In [30]:
# Construir datetime confiable por canal
sre_t = df[df["channel"] == "sre"].copy()
sre_t["dt"] = (
    pd.to_datetime(sre_t["ts"].astype(float), unit="s", utc=True)
    .dt.tz_convert("America/Mexico_City")
)

cx_t = df[df["channel"] == "monitoring-ops-cx"].copy()
cx_t["dt"] = (
    pd.to_datetime(cx_t["timestamp"], utc=True)
    .dt.tz_convert("America/Mexico_City")
)

sre_t["fecha"] = sre_t["dt"].dt.date
cx_t["fecha"]  = cx_t["dt"].dt.date

print("=== SRE: alertas por fecha ===")
print(sre_t.groupby("fecha").size().rename("n").to_frame().to_string())
print()
print("=== CX: alertas por fecha ===")
print(cx_t.groupby("fecha").size().rename("n").to_frame().to_string())

=== SRE: alertas por fecha ===
             n
fecha         
2025-05-26  21
2025-05-27  21
2025-05-28  15
2025-05-29  18
2025-05-30   2
2025-05-31  49
2025-06-01  70
2025-06-02   8
2025-06-03   6
2025-06-04   1
2025-06-07  25
2025-06-08  58
2025-06-09  47
2025-06-10  14
2025-06-11   8

=== CX: alertas por fecha ===
             n
fecha         
2026-03-24  57
2026-03-25  10
2026-03-26   5
2026-03-27  23


In [31]:
sre_t["hora"] = sre_t["dt"].dt.hour
cx_t["hora"]  = cx_t["dt"].dt.hour

print("=== SRE: alertas por hora del día (UTC-6, hora local CDMX/CST) ===")
sre_hora = sre_t.groupby("hora").size().rename("n").reindex(range(24), fill_value=0)
for h, n in sre_hora.items():
    bar = "█" * (n // 3)
    print(f"  {h:02d}h  {n:3d}  {bar}")

print()
print("=== CX: alertas por hora del día (UTC-6) ===")
cx_hora = cx_t.groupby("hora").size().rename("n").reindex(range(24), fill_value=0)
for h, n in cx_hora.items():
    if n > 0:
        bar = "█" * n
        print(f"  {h:02d}h  {n:3d}  {bar}")

=== SRE: alertas por hora del día (UTC-6, hora local CDMX/CST) ===
  00h    1  
  01h    4  █
  02h   24  ████████
  03h    0  
  04h    3  █
  05h    5  █
  06h    7  ██
  07h    5  █
  08h    4  █
  09h    5  █
  10h    2  
  11h   11  ███
  12h    9  ███
  13h   19  ██████
  14h   15  █████
  15h   24  ████████
  16h   39  █████████████
  17h   39  █████████████
  18h   44  ██████████████
  19h   46  ███████████████
  20h   25  ████████
  21h   16  █████
  22h   10  ███
  23h    6  ██

=== CX: alertas por hora del día (UTC-6) ===
  00h    1  █
  09h    1  █
  10h    1  █
  12h    3  ███
  13h   17  █████████████████
  14h   17  █████████████████
  15h   22  ██████████████████████
  16h   16  ████████████████
  19h    2  ██
  20h    2  ██
  22h    1  █
  23h   12  ████████████


In [32]:
# Alertas por día de semana (sre)
sre_t["dia_semana"] = sre_t["dt"].dt.day_name()
dias_orden = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
print("=== SRE: alertas por día de semana ===")
dow = sre_t["dia_semana"].value_counts().reindex(dias_orden, fill_value=0).rename("n").to_frame()
dow["pct"] = (dow["n"] / len(sre_t) * 100).round(1)
print(dow.to_string())

print()
# Horas pico por servicio (sre, top 5 servicios)
top5_svc = sre_t["service"].value_counts().head(5).index
print("=== Horas pico de los 5 servicios más activos (sre) ===")
for svc in top5_svc:
    sub = sre_t[sre_t["service"] == svc]
    hora_pico = sub["hora"].value_counts().idxmax()
    n_pico = sub["hora"].value_counts().max()
    horas_tipicas = sorted(sub["hora"].unique().tolist())
    print(f"  {svc:<25} pico={hora_pico:02d}h ({n_pico} alertas) | horas activas={horas_tipicas}")

=== SRE: alertas por día de semana ===
              n   pct
dia_semana           
Monday       76 20.90
Tuesday      41 11.30
Wednesday    24  6.60
Thursday     18  5.00
Friday        2  0.60
Saturday     74 20.40
Sunday      128 35.30

=== Horas pico de los 5 servicios más activos (sre) ===
  Orchestrator              pico=18h (16 alertas) | horas activas=[4, 7, 9, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22]
  Wallet_2.0                pico=16h (8 alertas) | horas activas=[14, 15, 16, 17, 18, 19, 20, 21]
  Hairs                     pico=19h (9 alertas) | horas activas=[1, 2, 6, 14, 15, 17, 18, 19, 20, 22, 23]
  tesseract                 pico=19h (9 alertas) | horas activas=[5, 13, 14, 15, 16, 17, 18, 19, 20, 21]
  Carts                     pico=02h (8 alertas) | horas activas=[2, 5, 6, 7, 14, 16, 17, 18, 19, 20, 22, 23]


---
## 13 · Hallazgos clave del EDA

Resumen de los descubrimientos más importantes para informar decisiones de pipeline.

In [33]:
hallazgos = [
    ("H-01", "Estructura",    "458 registros, 0 duplicados exactos. Lista JSON plana."),
    ("H-02", "Segmentos",     "3 exports concatenados: sre [0-151], sre [152-362], cx [363-457]. Períodos disjuntos."),
    ("H-03", "Esquemas",      "3 esquemas de llaves: sre-seg1/2 (10 cols), cx (12 cols +error_type/error_message), PayPal (sin priority/threshold/policy/incidents)."),
    ("H-04", "Relojes",       "ts en sre es epoch float confiable. ts en cx es sintético (gap=300 s). timestamp ISO en sre-seg2 tiene drift de -6h a +17h."),
    ("H-05", "Relojes",       "Regla operativa: sre → usar ts (epoch); cx → usar timestamp (ISO)."),
    ("H-06", "Nulos",         "363 nulos en error_type/error_message = ausencia estructural (solo cx). 3 nulos en priority/threshold/policy/incidents = registros PayPal Status."),
    ("H-07", "Cardinalidad",  "23 conditions únicas, 28 services, 5 policies (incl. null). condition→policy es 1-to-1. policy→channel es partición perfecta."),
    ("H-08", "Cardinalidad",  "Todos los conditions de cx están separados de sre. 0 conditions compartidos entre canales."),
    ("H-09", "incidents",     "Media 1.46, mediana 1, max 7. Suma total con null→1: 667 (664 solo New Relic)."),
    ("H-10", "threshold",     "26 reglas únicas. 92% estáticas, 8% anomalía/baseline. Dirección dominante: sobre (>). Ventana dominante: 5 min."),
    ("H-11", "Fingerprints",  "60 fingerprints únicos en raw. Top: Orchestrator::high request count w/ 500 (91 alertas, 19.9%). Varios singletones."),
    ("H-12", "cx",            "95 alertas cx, todas con error_type. Top: INSUFFICIENT_FUNDS 28 (29%), IMPOSSIBLE_TO_CHARGE 25 (26%), CARD_DECLINED 23 (24%)."),
    ("H-13", "cx",            "Procesador dominante en cx: Conekta. PayPal y Mercadopago tienen 1 registro cada uno de tipos raros."),
    ("H-14", "Temporal",      "sre cubre 17 días (26 may–11 jun 2025). cx cubre 4 días (24-27 mar 2026). Períodos NO solapados → no comparables."),
    ("H-15", "Temporal",      "Pico de alertas sre: 00h-02h (madrugada CDMX). Patrón nocturno en data-team (RDS) y Orchestrator."),
]

hall_df = pd.DataFrame(hallazgos, columns=["ID", "Categoría", "Hallazgo"])
pd.set_option("display.max_colwidth", 120)
hall_df

,ID,Categoría,Hallazgo
0,H-01,Estructura,"458 registros, 0 duplicados exactos. Lista JSON plana."
1,H-02,Segmentos,"3 exports concatenados: sre [0-151], sre [152-362], cx [363-457]. Períodos disjuntos."
2,H-03,Esquemas,"3 esquemas de llaves: sre-seg1/2 (10 cols), cx (12 cols +error_type/error_message), PayPal (sin priority/threshold/p..."
3,H-04,Relojes,ts en sre es epoch float confiable. ts en cx es sintético (gap=300 s). timestamp ISO en sre-seg2 tiene drift de -6h ...
4,H-05,Relojes,Regla operativa: sre → usar ts (epoch); cx → usar timestamp (ISO).
5,H-06,Nulos,363 nulos en error_type/error_message = ausencia estructural (solo cx). 3 nulos en priority/threshold/policy/inciden...
6,H-07,Cardinalidad,"23 conditions únicas, 28 services, 5 policies (incl. null). condition→policy es 1-to-1. policy→channel es partición ..."
7,H-08,Cardinalidad,Todos los conditions de cx están separados de sre. 0 conditions compartidos entre canales.
8,H-09,incidents,"Media 1.46, mediana 1, max 7. Suma total con null→1: 667 (664 solo New Relic)."
9,H-10,threshold,"26 reglas únicas. 92% estáticas, 8% anomalía/baseline. Dirección dominante: sobre (>). Ventana dominante: 5 min."
